# Language & GDP: Quantitative Regional Analysis
## Data Collection Phase (API-Driven, Pydantic Validated)

**Status:** Phase A - World Bank API only  
**Last Updated:** 2026-04-25  
**Architecture:** Pydantic models + API orchestrator

---

## 1. Setup & Dependencies

In [ ]:
# Install required packages
import subprocess
import sys

packages = [
    'pydantic>=2.0',
    'requests',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'scipy'
]

for package in packages:
    try:
        __import__(package.split('>=')[0].split('[')[0])
        print(f"✓ {package}")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
import sys
sys.path.insert(0, '.')

from data_pipeline import LanguageGDPDataPipeline
from models import RegionalGDPRecord, RegionalGDPDataset

# Style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.size'] = 10

print("✓ All modules loaded successfully")

## 2. Execute Data Pipeline

In [ ]:
# Initialize and run pipeline
pipeline = LanguageGDPDataPipeline('config.json')

# Run Phase A: World Bank API only
print("Executing pipeline...\n")
dataset = pipeline.run(sources=['world_bank'], use_cache=True)

## 3. Load and Inspect Data

In [ ]:
# Get DataFrame
df = pipeline.get_dataframe()

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nFirst 10 rows:")
print(df.head(10))

In [ ]:
# Data by language
print("\n=== Records by Language ===")
print(df.groupby('language').size())

print("\n=== Records by Country ===")
print(df.groupby('country').size())

print("\n=== Year Range ===")
print(f"From {df['year'].min()} to {df['year'].max()}")
print(f"\nYear distribution:")
print(df['year'].value_counts().sort_index(ascending=False).head(10))

## 4. Exploratory Data Analysis

In [ ]:
# Summary statistics by language (latest year available)
latest_year = df['year'].max()
df_latest = df[df['year'] == latest_year]

print(f"\n=== GDP Statistics by Language ({latest_year}) ===")
print(f"\nNote: National level (World Bank Phase A)\n")

summary = df_latest.groupby('language')['gdp_per_capita_usd'].agg([
    ('N', 'count'),
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('Std', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(0)

print(summary)

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box plot
ax1 = axes[0, 0]
sns.boxplot(data=df_latest, x='language', y='gdp_per_capita_usd', ax=ax1, palette='Set2')
ax1.set_ylabel('GDP per Capita (USD)')
ax1.set_title(f'GDP per Capita by Language ({latest_year})')
ax1.grid(True, alpha=0.3)

# Bar chart: Mean GDP/capita
ax2 = axes[0, 1]
means = df_latest.groupby('language')['gdp_per_capita_usd'].mean().sort_values(ascending=False)
ax2.bar(means.index, means.values, color=['#FF6B6B', '#4ECDC4', '#FFE66D'])
ax2.set_ylabel('Mean GDP per Capita (USD)')
ax2.set_title(f'Average GDP per Capita by Language')
ax2.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(means.values):
    ax2.text(i, v + 1000, f'${v:,.0f}', ha='center')

# Time series
ax3 = axes[1, 0]
for lang in df['language'].unique():
    lang_data = df[df['language'] == lang].groupby('year')['gdp_per_capita_usd'].mean()
    ax3.plot(lang_data.index, lang_data.values, marker='o', label=lang, linewidth=2)
ax3.set_xlabel('Year')
ax3.set_ylabel('Mean GDP per Capita (USD)')
ax3.set_title('GDP per Capita Trend Over Time')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Distribution by language
ax4 = axes[1, 1]
for lang in df_latest['language'].unique():
    lang_data = df_latest[df_latest['language'] == lang]['gdp_per_capita_usd']
    ax4.hist(lang_data, alpha=0.5, label=lang, bins=10)
ax4.set_xlabel('GDP per Capita (USD)')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of GDP per Capita')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_eda_by_language.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved as '01_eda_by_language.png'")

## 5. Statistical Tests (National Level)

In [ ]:
# ANOVA: Does language explain GDP variance?
print("\n=== ANOVA: Language Effect on GDP per Capita ===")
print(f"Data: Latest year ({latest_year})\n")

groups = [df_latest[df_latest['language'] == lang]['gdp_per_capita_usd'].values 
          for lang in df_latest['language'].unique()]

f_stat, p_value = stats.f_oneway(*groups)

print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.6f}")
if p_value < 0.05:
    print("✓ RESULT: Language SIGNIFICANTLY explains GDP variance (p < 0.05)")
else:
    print("✗ RESULT: Language does NOT significantly explain variance (p ≥ 0.05)")

# Effect size
grand_mean = df_latest['gdp_per_capita_usd'].mean()
ss_total = ((df_latest['gdp_per_capita_usd'] - grand_mean)**2).sum()
ss_between = sum(
    len(df_latest[df_latest['language'] == lang]) * 
    (df_latest[df_latest['language'] == lang]['gdp_per_capita_usd'].mean() - grand_mean)**2
    for lang in df_latest['language'].unique()
)
eta_squared = ss_between / ss_total if ss_total > 0 else 0

print(f"\nEffect Size (η²): {eta_squared:.4f}")
print(f"Interpretation: Language explains {eta_squared*100:.2f}% of variance")

In [ ]:
# Pairwise comparisons
from scipy.stats import ttest_ind

print("\n=== Pairwise T-Tests (National Level) ===")
languages = df_latest['language'].unique()

for i, lang1 in enumerate(languages):
    for lang2 in languages[i+1:]:
        data1 = df_latest[df_latest['language'] == lang1]['gdp_per_capita_usd'].values
        data2 = df_latest[df_latest['language'] == lang2]['gdp_per_capita_usd'].values
        
        t_stat, p_val = ttest_ind(data1, data2)
        mean_diff = data1.mean() - data2.mean()
        
        print(f"\n{lang1} vs {lang2}:")
        print(f"  Mean difference: ${mean_diff:,.0f}")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value: {p_val:.6f}")

## 6. Pipeline Status & Next Steps

In [ ]:
print("\n" + "="*70)
print("PIPELINE STATUS")
print("="*70)

print(f"\n✓ Phase A (National-level): COMPLETE")
print(f"  - Data source: World Bank API")
print(f"  - Countries: {', '.join(df['country'].unique())}")
print(f"  - Languages: {', '.join(df['language'].unique())}")
print(f"  - Year range: {df['year'].min()}-{df['year'].max()}")
print(f"  - Total records: {len(df)}")

print(f"\n⏳ Phase B (Regional-level): PLANNED")
print(f"  - German states: Destatis API (needs credentials)")
print(f"  - Swiss cantons: STATSWISS (file-based)")
print(f"  - Japanese prefectures: Cabinet Office (needs parsing)")
print(f"  - Spanish regions: INE API (in development)")

print(f"\n⏳ Phase C (With controls): PLANNED")
print(f"  - Add education spending (% of GDP)")
print(f"  - Add R&D intensity (% of GDP)")
print(f"  - Regression analysis with controls")

print(f"\n" + "="*70)
print(f"\nDataset exported to: ./data/processed/language_gdp_combined.csv")
print(f"Metadata exported to: ./data/processed/language_gdp_combined_metadata.json")

## 7. Code Architecture Overview

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║         ARCHITECTURE: Pydantic-Based API Pipeline            ║
╚════════════════════════════════════════════════════════════════╝

┌─────────────────────────────────────────────────────────────┐
│ models.py                                                    │
│  • RegionalGDPRecord (Pydantic validation)                  │
│  • RegionalGDPDataset (metadata + validation)               │
│  • APIFetchResult (tracking)                                │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ fetchers.py                                                  │
│  • BaseDataFetcher (abstract base class)                    │
│  • WorldBankDataFetcher (implemented)                       │
│  • DestatisDataFetcher (stub, ready for Phase B)            │
│  • [other fetchers...]                                      │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ data_pipeline.py                                             │
│  • LanguageGDPDataPipeline (orchestrator)                   │
│  • Error handling + logging                                 │
│  • Caching + export (CSV, JSON)                             │
└─────────────────────────────────────────────────────────────┘
                            ↓
                    [ This Notebook ]
                    (Analysis + Stats)

Key Features:
✓ Type-safe (Pydantic validation at every step)
✓ Extensible (add new API sources easily)
✓ Reproducible (API-driven, cached)
✓ Logged (detailed error tracking)
✓ Modular (separation of concerns)
""")

## References

- **World Bank API:** https://data.worldbank.org/docs/developer/overview/query-language
- **Pydantic v2:** https://docs.pydantic.dev/
- **Next Phase:** Regional-level data integration (Destatis, STATSWISS, etc.)

---
*Notebook: 01_data_collection.ipynb | Status: Phase A Complete*